# Our Stock Prediction Solution

**Strategy**: Predict `p50/p40` (the relative price change) using a Random Forest on 8 relative features, then rank stocks by expected dollar gain `p40 × (1 − predicted_ratio)` and sell the top 4,000.

**Why predict the ratio, not raw p50?**  
If we predict raw `p50`, the model learns that expensive stocks (high `p40`) tend to have high `p50` — a price-level bias. By predicting the *ratio* `p50/p40`, every feature becomes scale-free and the model focuses purely on the *direction* of price movement.

**Why rank by gain, not classify sell/no-sell?**  
R = Σ sell × (p40 − p50). We want to pick stocks with the *largest* drops, not just any drop. Ranking by magnitude directly maximises R.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PRICE_COLS = [f'p{i}' for i in range(1, 41)]  # p1 ... p40
MAX_SELL = 4000

## 1. Load & Clean Data

In [ ]:
train = pd.read_csv('train.csv', index_col='ID')
test  = pd.read_csv('test.csv',  index_col='ID')

# Drop the handful of rows with zero or missing p40 (can't compute a ratio)
train = train[train['p40'] > 0].dropna()

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head(3)

## 2. Quick EDA

Check the target `(p40 − p50)`: positive = profitable sell.  
Also check the ratio `p50/p40`: below 1 = price drops = good sell.

In [ ]:
gain  = train['p40'] - train['p50']
ratio = train['p50'] / train['p40']

print('=== (p40 - p50) gain distribution ===')
print(gain.describe().round(2))
print(f'\nStocks where p40 > p50 (profitable sell): {(gain > 0).sum()} / {len(gain)} ({(gain > 0).mean():.1%})')
print(f'Mean ratio p50/p40: {ratio.mean():.3f}  (below 1 = overall market drops on average)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gain, bins=80, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].axvline(0, color='red', lw=1.5, ls='--', label='breakeven')
axes[0].set_title('Distribution of Gain (p40 - p50)')
axes[0].set_xlabel('Gain from selling')
axes[0].legend()

axes[1].hist(ratio.clip(0, 2), bins=80, color='darkorange', alpha=0.8, edgecolor='none')
axes[1].axvline(1, color='red', lw=1.5, ls='--', label='ratio=1 (no change)')
axes[1].set_title('Distribution of p50/p40 Ratio')
axes[1].set_xlabel('p50 / p40')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Feature Engineering

All 8 features are **relative** (returns, ratios, normalised positions) so no stock's absolute price level biases the model.

| Feature | What it captures |
|---|---|
| `ret_full` | Log-return over all 40 days — long-term trend |
| `ret_20d` | Log-return over last 20 days — medium-term trend |
| `ret_10d` | Log-return over last 10 days — short-term momentum |
| `ret_5d` | Log-return over last 5 days — very recent momentum |
| `vol40` | Volatility (std of log-returns) over all 40 days |
| `vol10` | Volatility over last 10 days — is it getting noisier? |
| `pct_range` | Where is p40 within its 40-day high/low range? (0=at low, 1=at high) |
| `p40_vs_ma10_ratio` | p40 / 10-day average − 1 — is price above or below recent average? |

In [ ]:
def engineer_features(df):
    # Clip to avoid log(0); shape = (n_stocks, 40)
    p = np.clip(df[PRICE_COLS].values, 1e-6, None)

    # Log-returns between consecutive days; shape = (n_stocks, 39)
    log_rets = np.diff(np.log(p), axis=1)

    feats = {}

    # Momentum: log-return over different lookback windows
    feats['ret_full'] = np.log(p[:, -1] / p[:, 0])    # full 40-day return
    feats['ret_20d']  = np.log(p[:, -1] / p[:, 19])   # last 20 days
    feats['ret_10d']  = np.log(p[:, -1] / p[:, -11])  # last 10 days
    feats['ret_5d']   = np.log(p[:, -1] / p[:, -6])   # last 5 days

    # Volatility: std of daily log-returns
    feats['vol40'] = log_rets.std(axis=1)              # full window
    feats['vol10'] = log_rets[:, -10:].std(axis=1)    # recent window

    # Position within 40-day price range: 0 = at historical low, 1 = at historical high
    hist_min   = p.min(axis=1)
    hist_max   = p.max(axis=1)
    hist_range = hist_max - hist_min
    feats['pct_range'] = np.where(
        hist_range > 0,
        (p[:, -1] - hist_min) / hist_range,
        0.5
    )

    # p40 vs 10-day moving average (as a ratio − 1, so 0 = right at average)
    ma10 = p[:, -10:].mean(axis=1)
    feats['p40_vs_ma10_ratio'] = p[:, -1] / ma10 - 1

    return pd.DataFrame(feats, index=df.index)


X_train_feats = engineer_features(train)
X_test_feats  = engineer_features(test)

print(f'Feature matrix shape: {X_train_feats.shape}')
print(f'Any NaN: {X_train_feats.isna().any().any()}')
X_train_feats.head(3)

## 4. Build Regression Target

We predict `p50 / p40` (the ratio), not raw `p50`.  
- Ratio < 1 → price drops → good sell candidate  
- Ratio > 1 → price rises → don't sell  

We clip extreme outliers (ratio > 5) which are noise and would distort training.

In [ ]:
y_train = (train['p50'] / train['p40']).clip(upper=5)

print('Target (p50/p40) distribution:')
print(y_train.describe().round(3))
print(f'\nStocks with ratio < 1 (price drops): {(y_train < 1).sum()} / {len(y_train)} ({(y_train < 1).mean():.1%})')

## 5. Train / Validation Split

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_feats, y_train, test_size=0.2, random_state=RANDOM_STATE
)

# Keep actual p40 and p50 for scoring
p40_val = train.loc[X_val.index, 'p40']
p50_val = train.loc[X_val.index, 'p50']

print(f'Train: {X_tr.shape}  |  Val: {X_val.shape}')

## 6. Train Random Forest Regressor

Same Random Forest from the workshop — just `Regressor` instead of `Classifier` since we're predicting a continuous ratio.

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(X_tr, y_tr)

ratio_preds = model.predict(X_val)

# Convert predicted ratio back to p50 for RMSE reporting
val_p50_pred = p40_val.values * ratio_preds
rmse = mean_squared_error(p50_val, val_p50_pred) ** 0.5
mae  = mean_absolute_error(p50_val, val_p50_pred)

print(f'Validation RMSE (p50 equivalent): {rmse:.4f}')
print(f'Validation MAE  (p50 equivalent): {mae:.4f}')

## 7. Rank by Expected Gain → Evaluate Competition R

Expected gain for each stock = `p40 × (1 − predicted_ratio)`.  
We sort descending and pick the top K stocks (40% of holdout = allowed sells).

In [ ]:
def competition_score(p40, p50, ratio_preds, max_sell):
    expected_gain = p40.values * (1 - ratio_preds)

    # Sort by expected gain descending, pick top max_sell
    ranked_idx = np.argsort(expected_gain)[::-1]
    sell = np.zeros(len(ratio_preds), dtype=int)
    sell[ranked_idx[:max_sell]] = 1

    R = float((sell * (p40.values - p50.values)).sum())
    n_profitable = int((sell * (p40.values > p50.values)).sum())

    print(f'Stocks selected to sell:          {sell.sum()}')
    print(f'Of those, actually profitable:    {n_profitable} ({n_profitable/sell.sum():.1%})')
    print(f'Competition R:                    {R:.2f}')
    return R


K_val = int(0.4 * len(X_val))
R = competition_score(p40_val, p50_val, ratio_preds, K_val)

## 8. Feature Importance

A healthy model should spread importance across multiple features — no single feature should dominate.

In [ ]:
feat_imp = pd.Series(
    model.feature_importances_,
    index=X_train_feats.columns
).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
feat_imp.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(feat_imp.round(4))

## 9. Retrain on Full Training Data

Now that we've validated the model works, retrain on all ~10,000 rows to give the best possible predictions on the unseen test set.

In [ ]:
final_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

final_model.fit(X_train_feats, y_train)
print('Retrained on full training set.')

## 10. Generate Test Predictions & Submission

In [ ]:
test_ratio_pred  = final_model.predict(X_test_feats)
p40_test         = test['p40'].values
expected_gain_test = p40_test * (1 - test_ratio_pred)

# Rank and pick top 4000
ranked_idx = np.argsort(expected_gain_test)[::-1]
sell_test = np.zeros(len(test), dtype=int)
sell_test[ranked_idx[:MAX_SELL]] = 1

print(f'Total sells: {sell_test.sum()}')
print(f'Constraint satisfied (<=4000): {sell_test.sum() <= MAX_SELL}')

# Distribution check
plt.figure(figsize=(8, 4))
plt.hist(expected_gain_test, bins=80, color='steelblue', alpha=0.8, edgecolor='none')
plt.axvline(0, color='red', ls='--', lw=1.5, label='breakeven')
cutoff = expected_gain_test[ranked_idx[MAX_SELL - 1]]
plt.axvline(cutoff, color='orange', ls='--', lw=1.5, label=f'sell cutoff = {cutoff:.2f}')
plt.xlabel('Expected Gain (p40 × (1 − predicted ratio))')
plt.title('Test Set: Expected Gain Distribution')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
submission = pd.DataFrame({'ID': test.index, 'sell': sell_test})
submission.to_csv('our_submission.csv', index=False)

print('our_submission.csv saved')
print(submission['sell'].value_counts().rename({0: 'hold', 1: 'sell'}))
submission.head(10)